In [ ]:
import pandas as pd             # For handling data (DataFrame)
import seaborn as sns           # For visualizations and Planets dataset
import numpy as np              # For numerical operations
# import matplotlib.pyplot as plt # For plots
# import math

# from pandas import plotting

# import statsmodels.formula.api as smf
# from sklearn.linear_model import LinearRegression
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import mean_squared_error, r2_score

####Veh_Body_Type Notes, Explanations
"PASSENGER CAR" includes:
- Sedan, hatchback
- STATION WAGON (949)


"OTHER" includes:
- MOTORCYCLE - 3 WHEELED (3)
- LIMOUSINE (16)
- LOW SPEED VEHICLE (40)
- GOLF CART (3)
- AUTOCYCLE (50)
- RECREATIONAL OFF-HIGHWAY VEHICLES (ROV) (7)  
- FARM EQUIPMENT (TRACTOR, COMBINE HARVESTER, ETC.) (3)
- FARM VEHICLE (21)
- CONSTRUCTION EQUIPMENT (BACKHOE, BULLDOZER, ETC.) (25)
- SNOWMOBILE (130)

"VAN" assume Passenger (because otherwise specified)

####Note:  
For "CARGO VAN/LIGHT TRUCK 2 AXLES (OVER 10,000LBS (4,536 KG))",  
the weight is likely to be a mistake in the  
Automated Crash Reporting System (ACRS).

###Buses:
Buses are not grouped, because school bus circumstances are very  
different from local transit buses.  
Both are different from long-distance coaches, which are mostly on highways.

####Groupings for later:
Can group emergency vehicles in different ways.  
For now, want to explore the differences.

In [ ]:
crash_more_grouping_df = pd.read_csv(
    "https://raw.githubusercontent.com/Alissa-Ouspen/data201_alissa/main/final/crash_data_grouped_1.csv")


    #usecols = lambda x: x not in ["Second_Harmful_Event", "Driverless_Vehicle", "Veh_Going_Dir", "Route_Type",])

###crash_data_grouped_1.csv
--------------
Collision_Type, First_Harmful_Event, Driver_Distraction, and Junction remain,  
although probably won't use for current analysis.

###Working Here, Below


In [ ]:
crash_more_grouping_df.head()


,Vehicle_Num,Agency_Name,ACRS_Report_Type,Crash_Date_Time,Related_Non_Motorist,Collision_Type,Weather,Surface_Condition,Ambient_Light,Traffic_Control,...,Parked_Vehicle,Vehicle_Year,Hit_Run,Lane_Type,At_Fault,First_Harmful_Event,Junction,Intersection_Type,Road_Alignment,Road_Condition
0,0,Montgomery County Police,Injury Crash,5/5/26 22:43,NaN,Front to Front,Clear,Dry,Dark - Lighted,No Controls,...,No,2018.0,No,Lane 2,DRIVER,MOTOR VEHICLE IN TRANSPORT,NaN,NaN,Straight,No Defects
1,1,Montgomery County Police,Injury Crash,5/5/26 22:43,NaN,Front to Front,Clear,Dry,Dark - Lighted,No Controls,...,No,2026.0,No,Lane 2,DRIVER,MOTOR VEHICLE IN TRANSPORT,NaN,NaN,Straight,No Defects
2,2,Rockville Police Dept,Property Damage Crash,5/5/26 21:43,NaN,Front to Rear,Clear,Dry,Dark - Lighted,Flashing Traffic Control Signal,...,No,2023.0,No,Lane 2,NaN,MOTOR VEHICLE IN TRANSPORT,ACCELERATION/DECELERATION LANE,NaN,Straight,No Defects
3,3,Rockville Police Dept,Property Damage Crash,5/5/26 21:43,NaN,Front to Rear,Clear,Dry,Dark - Lighted,Flashing Traffic Control Signal,...,No,2005.0,No,Lane 2,NaN,MOTOR VEHICLE IN TRANSPORT,ACCELERATION/DECELERATION LANE,NaN,Straight,No Defects
4,4,Montgomery County Police,Property Damage Crash,5/5/26 21:30,NaN,Angle,Clear,Dry,Dark - Lighted,Lane Use Control Signal,...,No,2025.0,No,Lane 1,DRIVER,MOTOR VEHICLE IN TRANSPORT,CROSSOVER-RELATED,NaN,Straight,No Defects


In [ ]:
crash_more_grouping_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214273 entries, 0 to 214272
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Vehicle_Num                 214273 non-null  int64  
 1   Agency_Name                 214273 non-null  object 
 2   ACRS_Report_Type            214273 non-null  object 
 3   Crash_Date_Time             214273 non-null  object 
 4   Related_Non_Motorist        6767 non-null    object 
 5   Collision_Type              191162 non-null  object 
 6   Weather                     199566 non-null  object 
 7   Surface_Condition           189141 non-null  object 
 8   Ambient_Light               211494 non-null  object 
 9   Traffic_Control             182332 non-null  object 
 10  Driver_Substance_Use        170904 non-null  object 
 11  Non_Motorist_Substance_Use  5654 non-null    object 
 12  Driver_At_Fault             209591 non-null  object 
 13  Injury_Severit

In [ ]:
# Clean Junction: "Non-Junction" = "NON INTERSECTION" = "UNKNOWN" = nan
# Convert to uppercase for consistency
crash_culled_df["Junction"] = crash_culled_df["Junction"].astype(str).str.upper()

values_to_replace_junction = ["NON-JUNCTION", "NON INTERSECTION", "UNKNOWN", "NAN", "NA"]
crash_culled_df["Junction"] = crash_culled_df["Junction"].replace(values_to_replace_junction, np.nan)

print("Unique values after cleaning Junction:")
print(crash_culled_df["Junction"].unique())
print("\nValue counts for Junction:")
print(crash_culled_df["Junction"].value_counts(dropna=False))

Unique values after cleaning Junction:
[nan 'ACCELERATION/DECELERATION LANE' 'CROSSOVER-RELATED'
 'INTERSECTION OR RELATED' 'ENTRANCE/EXIT RAMP OR RELATED'
 'THROUGH ROADWAY' 'DRIVEWAY ACCESS OR RELATED'
 'OTHER LOCATION NOT LISTED ABOVE WITHIN AN INTERCHANGE AREA (MEDIAN,\nSHOULDER AND ROADSIDE)'
 'SHARED-USE PATH OR TRAIL' 'RAILWAY GRADE CROSSING' 'INTERSECTION'
 'INTERSECTION RELATED' 'OTHER DRIVEWAY' 'INTERCHANGE RELATED'
 'COMMERCIAL DRIVEWAY' 'CROSSOVER RELATED' 'RESIDENTIAL DRIVEWAY' 'ALLEY']

Value counts for Junction:
Junction
NaN                                                                                            102725
INTERSECTION                                                                                    62377
INTERSECTION RELATED                                                                            22185
INTERSECTION OR RELATED                                                                         12673
THROUGH ROADWAY                                   

In [ ]:
# Clean Collision_Type: 'other' = 'unknown' = nan
values_to_replace_ct = ["other", "unknown", "OTHER", "UNKNOWN"]
crash_culled_df["Collision_Type"] = crash_culled_df["Collision_Type"].astype(str).replace(values_to_replace_ct, np.nan, regex=True)

print("Unique values after cleaning Collision_Type:")
print(crash_culled_df["Collision_Type"].unique())
print("\nValue counts for Collision_Type:")
print(crash_culled_df["Collision_Type"].value_counts(dropna=False))

Unique values after cleaning Collision_Type:
['Front to Front' 'Front to Rear' 'Angle' 'Single Vehicle'
 'Sideswipe, Opposite Direction' 'nan' 'Rear To Side' 'Rear To Rear'
 'Sideswipe, Same Direction' 'OPPOSITE DIRECTION SIDESWIPE'
 'HEAD ON LEFT TURN' 'STRAIGHT MOVEMENT ANGLE' 'SAME DIR REAR END'
 'HEAD ON' 'SAME DIRECTION SIDESWIPE' 'SAME DIRECTION RIGHT TURN'
 'SAME DIRECTION LEFT TURN' 'ANGLE MEETS LEFT TURN'
 'SAME DIR BOTH LEFT TURN' 'SAME DIR REND LEFT TURN'
 'ANGLE MEETS RIGHT TURN' 'OPPOSITE DIR BOTH LEFT TURN'
 'ANGLE MEETS LEFT HEAD ON' 'SAME DIR REND RIGHT TURN']

Value counts for Collision_Type:
Collision_Type
SAME DIR REAR END                55759
STRAIGHT MOVEMENT ANGLE          30342
nan                              23111
Single Vehicle                   20818
SAME DIRECTION SIDESWIPE         16227
HEAD ON LEFT TURN                12926
Front to Rear                    12494
Angle                             8897
Sideswipe, Same Direction         6706
SAME DIRECTION RI

In [ ]:
# Clean First_Harmful_Event: Remove common 'unknown' / 'other' patterns
# Display unique values to identify cleaning needs, then apply standard replacements
print("Unique values of First_Harmful_Event before cleaning:")
print(crash_culled_df["First_Harmful_Event"].unique())

values_to_replace_fhe = ["unknown", "other", "UNKNOWN", "OTHER"]
crash_culled_df["First_Harmful_Event"] = crash_culled_df["First_Harmful_Event"].astype(str).replace(values_to_replace_fhe, np.nan, regex=True)

print("Unique values after cleaning First_Harmful_Event:")
print(crash_culled_df["First_Harmful_Event"].unique())
print("\nValue counts for First_Harmful_Event:")
print(crash_culled_df["First_Harmful_Event"].value_counts(dropna=False))

Unique values of First_Harmful_Event before cleaning:
['Motor Vehicle In Transport' 'Pedalcycle' 'Pedestrian' 'Fence'
 'Parked Vehicle' 'Curb' 'Tree (standing)' 'Other Post, Pole, Or Support'
 'Guardrail Face' 'Utility Pole/Light Support' 'Concrete Traffic Barrier'
 'Traffic Sign Support'
 'Other Fixed Object (wall, building, tunnel, etc.)'
 'Other Non-Fixed Object' 'Animal (live)' 'Other Non-Motorist'
 'Traffic Signal Support' 'Mailbox' 'Other Traffic Barrier' 'Ditch' nan
 'Overturn/Rollover' 'Fell Jumped from Motor Vehicle'
 'Other Non-Collision' 'Bridge Pier Or Support' 'Embankment'
 'Cable Barrier' 'Guardrail End Terminal'
 'Railway Vehicle (train, engine)' 'Culvert' 'Thrown Or Falling Object'
 'Bridge Rail' 'Explosion Or Fire' 'Spilled Cargo'
 'Strikes Object at Rest from Motor Vehicle In Transport'
 'Struck by Falling, Shifting Cargo Or Anything Set In Motion by Motor Vehicle'
 'Construction Equipment' 'Impact Attenuator/Crash Cushion'
 'Bridge Overhead Structure' 'Units Separate

In [ ]:
import numpy as np

# Rename the column first
crash_culled_df = crash_culled_df.rename(columns={"Road Alignment": "Road_Alignment"})

# Preserve original NaN values
original_nan_mask = crash_culled_df["Road_Alignment"].isna()

# Convert to string and title case for standardization, but only for non-NaN values
# This will turn np.nan into the string 'Nan'
crash_culled_df["Road_Alignment"] = crash_culled_df["Road_Alignment"].astype(str).str.title()

# Standardize 'Straight' values
straight_patterns = ["Straight", "Straight, Straight"]
crash_culled_df["Road_Alignment"] = crash_culled_df["Road_Alignment"].replace(straight_patterns, "Straight")

# Convert all variations of 'Curve' to 'Curved'
curve_patterns = [
    "Curve Left", "Curve Left, Straight", "Curve Right",
    "Curve Left, Curve Right", "Curve Right, Straight", "Curve Left, Curve Right, Straight"
]
crash_culled_df["Road_Alignment"] = crash_culled_df["Road_Alignment"].replace(curve_patterns, "Curved")

# Now, convert the string 'Nan' (which came from original np.nan) back to actual np.nan
crash_culled_df["Road_Alignment"] = crash_culled_df["Road_Alignment"].replace("Nan", np.nan)

# Ensure that any values that were originally NaN are still NaN after all processing
crash_culled_df.loc[original_nan_mask, "Road_Alignment"] = np.nan

print("Unique values after cleaning Road_Alignment:")
print(crash_culled_df["Road_Alignment"].unique())
print("\nValue counts for Road_Alignment:")
print(crash_culled_df["Road_Alignment"].value_counts(dropna=False))

Unique values after cleaning Road_Alignment:
['Straight' 'Curved']

Value counts for Road_Alignment:
Road_Alignment
Straight    173838
Curved       40435
Name: count, dtype: int64


In [ ]:
# Clean Vehicle_Damage: caseblind, "unkonwn" = "other" = nan
values_to_replace_vd = ["unkonwn", "other", "UNKNOWN", "OTHER"]
crash_culled_df["Vehicle_Damage"] = crash_culled_df["Vehicle_Damage"].astype(str).replace(values_to_replace_vd, np.nan, regex=True)

print("Unique values after cleaning Vehicle_Damage:")
print(crash_culled_df["Vehicle_Damage"].unique())
print("\nValue counts for Vehicle_Damage:")
print(crash_culled_df["Vehicle_Damage"].value_counts(dropna=False))

Unique values after cleaning Vehicle_Damage:
['DISABLING' 'FUNCTIONAL' 'VEHICLE NOT AT SCENE' 'SUPERFICIAL' 'NO DAMAGE'
 'NAN' 'DESTROYED']

Value counts for Vehicle_Damage:
Vehicle_Damage
DISABLING               79964
SUPERFICIAL             55186
FUNCTIONAL              54644
DESTROYED                7610
NAN                      7038
NO DAMAGE                6691
VEHICLE NOT AT SCENE     3140
Name: count, dtype: int64


In [ ]:
# Clean Veh_1st_Impact_Loc: "ROOF TOP" = "roof" = "top"; caseblind; OCLOCK = O Clock to O"Clock

# Convert to uppercase for case-insensitivity to standardize initial replacements
crash_culled_df["Veh_1st_Impact_Loc"] = crash_culled_df["Veh_1st_Impact_Loc"].astype(str).str.upper()

# Standardize "ROOF TOP", "ROOF", "TOP" to "ROOF TOP"
crash_culled_df["Veh_1st_Impact_Loc"] = crash_culled_df["Veh_1st_Impact_Loc"].replace(
    ["ROOF TOP", "ROOF", "TOP"], "ROOF TOP"
)

# Standardize OCLOCK, O CLOCK to O"CLOCK
crash_culled_df["Veh_1st_Impact_Loc"] = crash_culled_df["Veh_1st_Impact_Loc"].replace(
    ["OCLOCK", "O CLOCK"], "O\"CLOCK"
)

print("Unique values after cleaning Veh_1st_Impact_Loc:")
print(crash_culled_df["Veh_1st_Impact_Loc"].unique())
print("\nValue counts for Veh_1st_Impact_Loc:")
print(crash_culled_df["Veh_1st_Impact_Loc"].value_counts(dropna=False))

Unique values after cleaning Veh_1st_Impact_Loc:
['ONE O CLOCK' 'SIX O CLOCK' 'TWELVE O CLOCK' 'TWO O CLOCK'
 'VEHICLE NOT AT SCENE' 'ELEVEN O CLOCK' 'NINE O CLOCK' 'EIGHT O CLOCK'
 'SEVEN O CLOCK' 'FIVE O CLOCK' 'NON-COLLISION' 'TEN O CLOCK'
 'THREE O CLOCK' 'FOUR O CLOCK' 'UNDERSIDE' 'ROOF TOP' 'CARGO LOSS'
 'EIGHT OCLOCK' 'SEVEN OCLOCK' 'TWELVE OCLOCK' 'NAN' 'ONE OCLOCK'
 'SIX OCLOCK' 'TEN OCLOCK' 'FOUR OCLOCK' 'TWO OCLOCK' 'ELEVEN OCLOCK'
 'THREE OCLOCK' 'NINE OCLOCK' 'FIVE OCLOCK']

Value counts for Veh_1st_Impact_Loc:
Veh_1st_Impact_Loc
TWELVE OCLOCK           68025
SIX OCLOCK              34275
ONE OCLOCK              16197
ELEVEN OCLOCK           13774
TWELVE O CLOCK          12274
TWO OCLOCK               6238
TEN OCLOCK               6191
SIX O CLOCK              5818
ONE O CLOCK              4528
ELEVEN O CLOCK           4368
SEVEN OCLOCK             4279
FOUR OCLOCK              4126
FIVE OCLOCK              4064
EIGHT OCLOCK             3700
THREE OCLOCK             3379
N

In [ ]:
# Clean Vehicle_Damage: caseblind, "unkonwn" = "other" = nan
values_to_replace_vd = ["unkonwn", "other"]
crash_culled_df["Vehicle_Damage"] = crash_culled_df["Vehicle_Damage"].astype(str).replace(values_to_replace_vd, np.nan, regex=True)
# Convert to uppercase for case-insensitivity, then replace
crash_culled_df["Vehicle_Damage"] = crash_culled_df["Vehicle_Damage"].astype(str).str.upper().replace(["UNKNOWN", "OTHER"], np.nan)

print("Unique values after cleaning Vehicle_Damage:")
print(crash_culled_df["Vehicle_Damage"].unique())
print("\nValue counts for Vehicle_Damage:")
print(crash_culled_df["Vehicle_Damage"].value_counts(dropna=False))

Unique values after cleaning Vehicle_Damage:
['DISABLING' 'FUNCTIONAL' 'VEHICLE NOT AT SCENE' 'SUPERFICIAL' 'NO DAMAGE'
 'NAN' 'DESTROYED']

Value counts for Vehicle_Damage:
Vehicle_Damage
DISABLING               79964
SUPERFICIAL             55186
FUNCTIONAL              54644
DESTROYED                7610
NAN                      7038
NO DAMAGE                6691
VEHICLE NOT AT SCENE     3140
Name: count, dtype: int64


In [ ]:
# Clean Intersection_Type
crash_culled_df["Intersection_Type"] = crash_culled_df["Intersection_Type"].replace(
    ["ROUNDABOUT", "TRAFFIC CIRCLE"], "Roundabout/Traffic Circle"
)

print("Unique values after cleaning Intersection_Type:")
print(crash_culled_df["Intersection_Type"].unique())


Unique values after cleaning Intersection_Type:
[nan 'Perpendicular' 'Angled/Skewed' 'Roundabout/Traffic Circle'
 'FOUR-WAY INTERSECTION' 'T-INTERSECTION' 'FIVE-POINT OR MORE'
 'Y-INTERSECTION']



Drivers_License_State  change
"XX" to nan,
 change "MX-MEX", "MX-MX-ROO", "MX-GRO" to "Mex",
change CA-ON, CA-QC, ON, QC, BC, and any other codes that are canadian provinces to "Can"

To be cleaned up now:  

- "Intersection_Type"  

- Lane_Type to Parking_Lot Y/N/nan  
check against off road

- Vehicle_Damage:  caseblind, "unkonwn" = "other" = nan

Veh_1st_Impact_Loc : "ROOF TOP" = "roof" = "top"; caseblind; OCLOCK = O Clock to O'Clock

- Driver_Distraction:  Unknown = UNKNOWN = "NO DRIVER PRESENT" nan

- "Collision_Type"  "other" = "unknown" = nan

- Veh_Body_Type : caseblind,  unknown = other,

- First_Harmful_Event :   
- Second_Harmful_Event :  

- Junction : "Non-Junction" =  "NON INTERSECTION" = "UNKNOWN"  = nan

- Intersection_Type :   
"Roundabout/Traffic Circle" = "ROUNDABOUT" = "TRAFFIC CIRCLE"

- Road Alignment: if not Straight or nan then change to curved



####Categorical Data:
ACRS_Report_Type : ['Injury Crash' 'Property Damage Crash' 'Fatal Crash']



####Date/Time:
Date_Time

####Clock Positions and Crash Impact
Twelve o'clock = front bumper  
Three  o'clock = right (US passenger) side center  
Six o'clock = rear bumper  
Nine o'clock = left (US driver) side center  

Alaska Department of Public Safety  
https://tracs.dps.alaska.gov/TechSupport/12-200.1/MotorVehicle/DamagedAreas.htm

------
####Driver_Distraction  

these categories are good.  
Optional to later group into  
Device-related,  
Passenger-related  
Vehicle device controls  
etc.


---------------
####Future research:  
It would be useful to interview representatives of the municipalities about how they classify and record specific details about crashes, such as drug and alcohol use.  
I would also like to know why the handbook lists many more fields in covered by the Automated Crash Reporting System (ACRS)

Maryland Department of State Police
https://mdsp.maryland.gov/Pages/Dashboards/CrashDataDashboard.aspx  
Maryland Crash Data 2024-Present
Map  
Uses updated information, but that information is not shared with the public (only the preliminary information, which may be incomplete or contain errors for the reasons listed).

Age of driver

Include data about passengers:  number of passengers, general age
------------

####Note on Data
In trying to understand the meaning of certain fields, I may have found an error in the Incidents data description on  
https://data.montgomerycountymd.gov/Public-Safety/Crash-Reporting-Incidents-Data/bhju-22kf/about_data  
The "Direction" field is described as "Location - Direction from mile point."  
The "Distance" field is described as "Location - Distance from mile point."  

However, the Maryland Automated Crash Reporting System (ACRS) Field Reference Guide published on  
https://www.nhtsa.gov/sites/nhtsa.gov/files/documents/acrsfieldreference.pdf
(2017 Maryland State Police)  
section 2.19 states:  
"Distance
Definition:
The distance from the referenced Intersecting Road to the crash site.
Explanation:
The distance in feet or miles from the Intersecting Road to crash site."

Section 2.21:  
"Distance Direction
Definition:
The compass direction describing the direction going from the primary and intersecting
roads to the crash.
Explanation:
Units are described in compass points, i.e. North, South East and West. If the crash
occurred in the intersection, select the compass point referenced in the Mile Point
Direction box above."
---------------------


####About Data

Crash Reporting - Incidents Data  
Original data:  
122,000 Rows  
37 Columns  
Each row is a "Collision"

https://data.montgomerycountymd.gov/Public-Safety/Crash-Reporting-Incidents-Data/bhju-22kf/about_data  

"This dataset provides general information about each collision and details of all traffic collisions occurring on county and local roadways within Montgomery County, as collected via the Automated Crash Reporting System (ACRS) of the Maryland State Police, and reported by the Montgomery County Police, Gaithersburg Police, Rockville Police, or the Maryland-National Capital Park Police.

Please note that these collision reports are based on preliminary information supplied to the Police Department by the reporting parties. Therefore, the collision data available on this web page may reflect:

-Information not yet verified by further investigation  
-Information that may include verified and unverified collision data  
-Preliminary collision classifications may be changed at a later date based   upon further investigation  
-Information may include mechanical or human error"

"3.38 Going Direction
Definition:
The direction of the motor vehicles travel on the roadway prior to the crash.
Explanation:
This is not necessarily a compass direction, but must be consistent with the "Distance
Direction" from the Log Mile Information for this roadway."


![crash map disclaimer](https://raw.githubusercontent.com/Alissa-Ouspen/data201_alissa/main/final/crash_map_disclaimer.png)




####Sources:

- Learn types of intersections | Complete Guide to 12 Types  
https://www.dmvpermittests.com/blog/different-types-of-intersections

IGNORE

AI usage tracker:

- For all cells in merged_cleaned_df:
if value in cell == "unkonwn" or "UNKNOWN" or "other" or "OTHER" or "Not Applicable" or "NOT APPLICABLE",  
set value to nan  

- Make all caseblind:  
Compare values in each column.  
If a value written in all caps has a matching value that is written in mixed case or lower case,  
then convert all data written in all caps into the matching version that is written in mixed case or lower case.

- Drivers_License_State change "XX" to nan, change "MX-__" to "Mex", change CA-ON, CA-QC, ON, QC, BC, and any other codes that are canadian provinces to "Can".  List all US states and DC.  List all US territories and create Territories group. Create Foreign/Other group for all else except nan.  

- Veh_1st_Impact_Loc
"ROOF TOP", "roof" : "top"   
OCLOCK, O Clock : O'Clock

